## t3

In [1]:
import numpy as np
from collections import defaultdict
# 定义常量
WEATHER_LIST = [
    '晴朗', '高温', '晴朗', '晴朗', '晴朗', '晴朗', '高温', '高温', '高温','高温'
]

MINE_NODES = [55]
VILLAGE_NODES = []
DESTINATION_NODES = [13]

END_NODE = 13
MAX_DAY = 10
INITIAL_MONEY = 10000
WEIGHT_LIMIT = 1200
BASE_INCOME = 200

# 资源参数
WATER_WEIGHT = 3
FOOD_WEIGHT = 2
WATER_PRICE = 5
FOOD_PRICE = 10
WATER_REFUND = 2.5
FOOD_REFUND = 5

# 基础消耗量 (晴朗, 高温, 沙暴)
BASE_CONSUMPTION = {
    '晴朗': (3, 4),
    '高温': (9, 9),
    '沙暴': (10, 10)
}

# 地图邻接表
GRAPH = {
    1: [2, 4, 5],
    2: [3, 4, 1],
    3: [2, 4, 8, 9],
    4: [1, 2, 3, 5, 6, 7],
    5: [1, 4, 6],
    6: [4, 5, 7, 12, 13],
    7: [4, 6, 11, 12],
    8: [3, 9],
    9: [3, 8, 10, 11],
    10: [9, 11, 13],
    11: [7, 9, 10, 12, 13],
    12: [7, 11, 13]
}

# 运动惯性路径定义
MOTION_INERTIA_PATHS = {
    # # 路径1: 23-21-9-15-13-12
    # (23, 21): 9,
    # (21, 9): 15,
    # (9, 15): 13,
    # (15, 13): 12,
    # # 路径2: 12-13-15 (反向)
    # (12, 13): 15,
    # (13, 15): 9,  # 15可以选择去9或13
    # # 路径3: 15-9-21-27
    # (15, 9): 21,
    # (9, 21): 27
}

# 各节点到终点的最短天数
SHORTEST_DAYS_TO_END = {
    1: 3,
    2: 3,
    3: 3,
    4: 2,
    5: 2,
    6: 1,
    7: 2,
    8: 3,
    9: 2,
    10: 1,
    11: 1,
    12: 1
}

In [ ]:
NUM_PLAYERS = 2
# 对手路径 (day 0 to 9)
OPPONENT_PATH_1 = [1, 4, 4, 6, 13, 13, 13, 13, 13, 13]
# OPPONENT_PATH_2 = [1, 5, 5, 6, 13, 13, 13, 13, 13, 13]
# OPPONENT_PATH_3 = [1, 1, 1, 4, 6, 13, 13, 13, 13, 13]
# OPPONENT_PATH_4 = [1, 1, 1, 5, 6, 13, 13, 13, 13, 13]
# OPPONENT_PATH_5 = [1, 1, 1, 1, 5, 6, 13, 13, 13, 13]
# OPPONENT_PATH_6 = [1, 1, 1, 1, 4, 6, 13, 13, 13, 13]

OPPONENT_PATHS = [OPPONENT_PATH_1]#, OPPONENT_PATH_2, OPPONENT_PATH_3, OPPONENT_PATH_4, OPPONENT_PATH_5, OPPONENT_PATH_6]

def exponential_probabilities(base=2, normalize=True):
    """
    使用指数函数生成递减概率
    
    Args:
        base: 指数底数，越大递减越快
        normalize: 是否归一化
    """
    # 生成指数权重：base^0, base^0, base^(-1), base^(-1), base^(-2), base^(-2)
    # exponents = [0, 0, -1, -1, -2, -2]
    weights = [1]
    
    if normalize:
        total_weight = sum(weights)
        probabilities = [w / total_weight for w in weights]
        return probabilities
    else:
        return weights

# 使用指数函数
OPPONENT_PATH_PROB = exponential_probabilities(base=2)

In [ ]:
def calculate_next_move_need(day, count):
    """
    计算从指定日期开始，接下来指定次数移动所需的总物资（水和食物）。

    Args:
        day (int): 开始计算的日期 (1-indexed)。
        move_count (int): 移动次数。

    Returns:
        tuple: (总共需要的水, 总共需要的食物)。
               如果无法完成指定次数移动（例如，日期超出范围），则返回None。
    """
    total_water_needed = 0
    total_food_needed = 0
    moves_count = 0
    current_day_index = day + 1   # 调整为day之后一天

    while moves_count < count:
        if current_day_index >= len(WEATHER_LIST):
            # 如果日期超出天气列表范围，则无法完成3次移动
            return total_water_needed, total_food_needed

        weather = WEATHER_LIST[current_day_index]
        base_water, base_food = BASE_CONSUMPTION[weather]

        if weather == '沙暴':
            # 沙暴天气，停留，消耗基础物资
            total_water_needed += base_water
            total_food_needed += base_food
        else:
            # 非沙暴天气，移动，消耗双倍物资
            total_water_needed += base_water * 2
            total_food_needed += base_food * 2
            moves_count += 1
        
        current_day_index += 1
        
    return total_water_needed, total_food_needed

In [ ]:
def initialize_dp():
    dp = [dict() for _ in range(MAX_DAY + 1)]
    # 第0天：在起点购买资源
    for food in range(0, 180):
        for water in range(0, 195):
            # if food < water: continue
            weight = WATER_WEIGHT * water + FOOD_WEIGHT * food
            
            # 无法再多带一箱水或食物
            if (weight > WEIGHT_LIMIT):
                continue
            
            cost = WATER_PRICE * water + FOOD_PRICE * food
            if cost <= INITIAL_MONEY:
                money = INITIAL_MONEY - cost
                state = (1, water, food)
                dp[0][state] = (money, None, None, None, None)
    return dp
    
def get_base_consumption(weather):
    return BASE_CONSUMPTION[weather]

def update_state(dp_next, state, money, pre_pos, pre_money, pre_water, pre_food):
    pos, water, food = state
    # 检查资源非负
    if water < 0 or food < 0:
        return
    # 检查负重限制
    weight = WATER_WEIGHT * water + FOOD_WEIGHT * food
    if weight > WEIGHT_LIMIT:
        return
    # 更新状态：保留资金最大的
    if state in dp_next:
        if money > dp_next[state][0]:
            dp_next[state] = (money, pre_pos, pre_money, pre_water, pre_food)
    else:
        dp_next[state] = (money, pre_pos, pre_money, pre_water, pre_food)

In [ ]:
def _calculate_future_max_consumption():
    """
    Pre-calculates the maximum water and food needed from each day until the end.
    The consumption is estimated at 3 times the base rate for any weather.
    """
    future_max_consumption = {}
    # Start from the last day and go backwards
    max_needed_water = 0
    max_needed_food = 0
    for day in range(MAX_DAY - 1, -1, -1):
        # For day `d`, we need to calculate consumption for days `d+1` to `MAX_DAY-1`
        # The values for day `d` are the same as for `d+1` plus consumption on day `d+1`
        # But since we iterate backwards, we can just accumulate
        future_max_consumption[day] = (max_needed_water, max_needed_food)
        
        # Add consumption for the current day to be used for the previous day's calculation
        weather = WEATHER_LIST[day]
        bw, bf = get_base_consumption(weather)
        max_needed_water += 3 * bw
        max_needed_food += 3 * bf
        
    return future_max_consumption

# Pre-calculate the values once when the module is imported
FUTURE_MAX_CONSUMPTION = _calculate_future_max_consumption()
#FUTURE_MAX_CONSUMPTION


In [ ]:
# dp = initialize_dp()
# dp

In [ ]:
def get_opponent_position_probability(day):
    """
    获取指定天数另一个玩家在各个位置的概率分布
    """
    prob_dist = {}
    for i, path in enumerate(OPPONENT_PATHS):
        if day < len(path):
            pos = path[day]
        else:
            # 超出路径长度后，停留在最后一个位置
            pos = path[-1]
        
        if pos in prob_dist:
            prob_dist[pos] += OPPONENT_PATH_PROB[i]
        else:
            prob_dist[pos] = OPPONENT_PATH_PROB[i]
    
    return prob_dist

def calculate_expected_consumption(base_water, base_food, pos, opponent_prob_dist, action_type):
    """
    计算考虑另一个玩家影响后的期望消耗
    action_type: 'move', 'mine'
    """
    # 获取当前位置另一个玩家存在的概率
    opponent_prob_at_pos = opponent_prob_dist.get(pos, 0)
         
    if action_type == 'move' and pos != END_NODE:
        # 移动：2倍基础消耗
        expected_water = 2 * base_water
        expected_food = 2 * base_food
        # 如果另一个玩家在同一区域移动，消耗可能受影响
        if opponent_prob_at_pos > 0:
            expected_water = int(2 * expected_water * opponent_prob_at_pos + expected_water * (1 - opponent_prob_at_pos))
            expected_food = int(2 * expected_food * opponent_prob_at_pos + expected_food * (1 - opponent_prob_at_pos))
            
    elif action_type == 'mine':
        # 挖矿：额外2倍消耗
        expected_water = base_water + 2 * base_water  # 基础 + 额外
        expected_food = base_food + 2 * base_food
        # 如果另一个玩家在同一矿山，收益可能受影响
        expected_income = BASE_INCOME
        if opponent_prob_at_pos > 0:
            # 根据题目描述，同一矿山时收益为基础的1/k倍
            # 这里假设k=2（两个玩家）
            expected_income = BASE_INCOME / NUM_PLAYERS
        return expected_water, expected_food, expected_income
    
    return expected_water, expected_food


In [61]:
def simulate_two_players():
    """
    双人游戏模拟函数
    """
    dp = initialize_dp()
    global_max_money = -1
    best_strategy = []
    
    for day in range(MAX_DAY):
        print(f"第{day+1}天")
        
        weather = WEATHER_LIST[day]
        base_water, base_food = get_base_consumption(weather)
        
        # 获取当天另一个玩家的位置概率分布
        opponent_prob_dist = get_opponent_position_probability(day)
        #print(f"对手位置概率分布: {opponent_prob_dist}")
        
        dp_next = dict()
        
        for state, (money, pre_pos, pre_money, pre_water, pre_food) in dp[day].items():
            pos, water, food = state
            if pos in DESTINATION_NODES: 
                continue

            # 剪枝：如果剩余天数不足以到达终点
            if (day) + SHORTEST_DAYS_TO_END.get(pos, float('inf')) > MAX_DAY:
                continue
            
            # 停留
            new_pos = pos
            new_water = water - base_water
            new_food = food - base_food
            new_money = money
            update_state(dp_next, (new_pos, new_water, new_food), new_money, pos, money, water, food)

            # 挖矿选项（如果在矿山）
            if pos in MINE_NODES:
                mine_water, mine_food, mine_income = calculate_expected_consumption(
                    base_water, base_food, pos, opponent_prob_dist, 'mine'
                )
                
                mine_water_final = water - mine_water
                mine_food_final = food - mine_food
                mine_money_final = money + mine_income
                update_state(dp_next, (new_pos, mine_water_final, mine_food_final), 
                           mine_money_final, pos, money, water, food)
            
            # 选项2: 行走到相邻节点
            for neighbor in GRAPH[pos]:
                
                # 计算移动的期望消耗
                expected_water, expected_food = calculate_expected_consumption(
                    base_water, base_food, pos, opponent_prob_dist, 'move'
                )
                
                new_pos = neighbor
                new_water = water - expected_water
                new_food = food - expected_food
                new_money = money
                
                # 检查是否到达终点
                if new_pos == END_NODE and new_water >= 0 and new_food >= 0 and new_money >= 0:
                    final_money = new_money + WATER_REFUND * new_water + FOOD_REFUND * new_food
                    if final_money > global_max_money:
                        global_max_money = final_money 
                        print(f"新的最大期望收益: {global_max_money}")
                        print(f"在第{day+1}天到达终点")
                        print(f"最终状态: 位置={new_pos}, 资金={new_money}, 水={new_water}, 食物={new_food}")
                    update_state(dp_next, (new_pos, new_water, new_food), new_money, pos, money, water, food)
                else:
                    update_state(dp_next, (new_pos, new_water, new_food), new_money, pos, money, water, food)

        dp[day + 1] = dp_next
    
    return global_max_money, dp

In [64]:
max_money, dp = simulate_two_players()
print(f"最大收益: {max_money}")
# print(f"dp: {dp}")



第1天
第2天
第3天
新的最大期望收益: 9340.0
在第3天到达终点
最终状态: 位置=13, 资金=9340, 水=0, 食物=0
第4天
新的最大期望收益: 9360.0
在第4天到达终点
最终状态: 位置=13, 资金=9360, 水=0, 食物=0
第5天
新的最大期望收益: 9405.0
在第5天到达终点
最终状态: 位置=13, 资金=9405, 水=0, 食物=0
第6天
第7天
第8天
第9天
第10天
最大收益: 9405.0


In [ ]:
day = 5
# 这是您要查询的初始 key
current_key = (13, 0, 0) 

# 检查 dp 表和初始 key 是否存在
if not 'dp' in locals() or not isinstance(dp, list) or len(dp) <= day:
    print("错误: 'dp' 列表不存在或不完整。请先运行您的模拟代码来填充 'dp' 表。")
elif current_key not in dp[day]:
    print(f"错误: 在 dp[{day}] 中找不到初始状态 {current_key}。")
else:
    # 获取最终状态的 value
    current_value = dp[day][current_key]
    
    # 用于存储路径
    path = [(current_key, current_value)]

    # 从第 23 天开始向前回溯
    for d in range(day - 1, -1, -1):
        # 根据您描述的逻辑，从当前 value 构建前一天的 key
        # value[1] 是前一天的 pos
        # value[3] 是前一天的 water
        # value[4] 是前一天的 food
        # 注意：这里的索引基于您图片中 value 的结构 (money, pre_pos, pre_money, pre_water, pre_food)
        prev_key = (current_value[1], current_value[3], current_value[4])
        
        # 在前一天的 dp 表中查找
        if prev_key in dp[d]:
            # 找到了，更新当前状态并记录路径
            current_key = prev_key
            current_value = dp[d][current_key]
            path.append((current_key, current_value))
        else:
            # 如果找不到，说明路径中断
            print(f"路径在第 {d} 天中断，无法找到状态: {prev_key}")
            break
            
    # 打印完整的回溯路径（从开始到结束）
    print("--- 最优路径回溯结果 ---")
    # 我们从后往前找的，所以需要反转列表
    for i, (key, value) in enumerate(reversed(path)):
        # 这里的天数是模拟的天数（从1开始）
        day_num = day - len(path) + i + 2
        print(f"第 {day_num-1} 天: 状态={key}, 值={value}")
        

--- 最优路径回溯结果 ---
第 0 天: 状态=(1, 33, 43), 值=(9405, None, None, None, None)
第 1 天: 状态=(1, 30, 39), 值=(9405, 1, 9405, 33, 43)
第 2 天: 状态=(1, 21, 30), 值=(9405, 1, 9405, 30, 39)
第 3 天: 状态=(4, 13, 19), 值=(9405, 1, 9405, 21, 30)
第 4 天: 状态=(6, 7, 10), 值=(9405, 4, 9405, 13, 19)
第 5 天: 状态=(13, 0, 0), 值=(9405, 6, 9405, 7, 10)


: 

In [26]:
import csv

# --- 您提供的代码开始 ---
day = 30
# 这是您要查询的初始 key
current_key = (64, 0, 0)

# 检查 dp 表和初始 key 是否存在
if not 'dp' in locals() or not isinstance(dp, list) or len(dp) <= day:
    print("错误: 'dp' 列表不存在或不完整。请先运行您的模拟代码来填充 'dp' 表。")
elif current_key not in dp[day]:
    print(f"错误: 在 dp[{day}] 中找不到初始状态 {current_key}。")
else:
    # 获取最终状态的 value
    current_value = dp[day][current_key]
    
    # 用于存储路径
    path = [(current_key, current_value)]

    # 从第 29 天开始向前回溯
    for d in range(day - 1, -1, -1):
        # 根据您描述的逻辑，从当前 value 构建前一天的 key
        # value[1] 是前一天的 pos
        # value[3] 是前一天的 water
        # value[4] 是前一天的 food
        prev_key = (current_value[1], current_value[3], current_value[4])
        
        # 在前一天的 dp 表中查找
        if prev_key in dp[d]:
            # 找到了，更新当前状态并记录路径
            current_key = prev_key
            current_value = dp[d][current_key]
            path.append((current_key, current_value))
        else:
            # 如果找不到，说明路径中断
            print(f"路径在第 {d} 天中断，无法找到状态: {prev_key}")
            break
    
    # --- 修改部分：将结果写入 CSV 文件 ---
    
    output_filename = 'optimal_path_trace.csv'
    print(f"--- 正在将最优路径回溯结果写入到 {output_filename} ---")

    try:
        with open(output_filename, 'w', newline='', encoding='utf-8') as csvfile:
            # 创建 CSV 写入器
            csv_writer = csv.writer(csvfile)
            
            # 写入表头
            header = [
                '天数', '位置', '水量', '食物量', 
                '当前金钱', '前一天位置', '前一天金钱', 
                '前一天水量', '前一天食物量'
            ]
            csv_writer.writerow(header)
            
            # 我们从后往前找的，所以需要反转列表以按天数顺序写入
            for i, (key, value) in enumerate(reversed(path)):
                # 计算天数
                day_num = day - len(path) + i + 1
                
                # 分解 key 和 value 以便写入
                pos, water, food = key
                money, pre_pos, pre_money, pre_water, pre_food = value
                
                # 准备要写入的一行数据
                row_data = [
                    day_num, pos, water, food,
                    money, pre_pos, pre_money,
                    pre_water, pre_food
                ]
                
                # 写入数据行
                csv_writer.writerow(row_data)
        
        print(f"--- 成功写入 {output_filename} ---")

    except IOError as e:
        print(f"写入文件时出错: {e}")


--- 正在将最优路径回溯结果写入到 optimal_path_trace.csv ---
--- 成功写入 optimal_path_trace.csv ---
